In [1]:
import pandas as pd
import numpy as np
import torch

from transformer.vocabs import get_or_create_smiles_vocabs
from transformer.data_funcs import calculate_max_mz
from transformer.transformer_utils import split_and_load_tokenized_multimodal_data
from transformer.data_funcs import bin_spectrum
from transformer.models import MultimodalVITSeq2SeqBeam
from transformer.sequence_pred import (
    prepare_dataframes_for_prediction, 
    prepare_spectra_for_prediction, 
    predict_smiles_from_spectra
)

In [2]:
if torch.cuda.is_available():
    print("CUDA is available.")
    print("PyTorch version:", torch.__version__)
    print("CUDA version:", torch.version.cuda)
    print("Number of available GPUs:", torch.cuda.device_count())
    print("GPU name:", torch.cuda.get_device_name(0))
else:
    print("CUDA is not available.")

CUDA is available.
PyTorch version: 2.0.1+cu118
CUDA version: 11.8
Number of available GPUs: 1
GPU name: NVIDIA GeForce RTX 3080


In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [4]:
ir = pd.read_feather('data/ir_spectra_v2.feather')
ir.columns= ['SMILES', 'spectrum']

In [5]:
ir.head()

,SMILES,spectrum
0,COc1nc2ccccc2cc1C(=O)O,"[-0.004481, 0.007279, -0.006439, 0.007464, -0...."
1,CCOC(=O)c1cc2c(OCc3coc4cc(F)ccc34)cccc2n1C(=O)...,"[0.028195, 0.011433, 0.024865, -0.00816, 0.055..."
2,CCCCOc1c(CN2C(=O)c3ccccc3C2=O)n(CC2CC2)c(=O)c2...,"[-0.003043, 0.014693, -0.003067, 0.025262, -0...."
3,CCCCNC[C@@H]1O[C@](O)(CO)[C@@H](O)[C@@H]1O,"[0.099321, 0.155452, 0.048156, 0.19118, 0.0237..."
4,O=C(NCC1CC1)c1nc2c(N3CCC(n4c(=O)[nH]c5ccccc54)...,"[0.2335, 0.090925, 0.061724, 0.158822, 0.04248..."


In [6]:
ms = pd.read_feather('data/msms_cfmid_positive_40ev_v2.feather')
ms.columns = ['SMILES', 'spectrum']

In [7]:
ms.head()

,SMILES,spectrum
0,COc1nc2ccccc2cc1C(=O)O,"[[101.03858, 100.0], [103.05423, 43.72], [104...."
1,CCOC(=O)c1cc2c(OCc3coc4cc(F)ccc34)cccc2n1C(=O)...,"[[57.06988, 51.14], [91.05423, 9.59], [121.044..."
2,CCCCOc1c(CN2C(=O)c3ccccc3C2=O)n(CC2CC2)c(=O)c2...,"[[57.06988, 28.12], [105.03349, 100.0], [117.0..."
3,CCCCNC[C@@H]1O[C@](O)(CO)[C@@H](O)[C@@H]1O,"[[30.03383, 36.99], [41.03858, 8.43], [42.0338..."
4,O=C(NCC1CC1)c1nc2c(N3CCC(n4c(=O)[nH]c5ccccc54)...,"[[55.05423, 63.08], [70.02874, 18.67], [70.065..."


In [8]:
# Minimal test
n = 50000
df_ir = ir.sample(n, random_state=42)
df_ir_test = ir.drop(index=df_ir.index).sample(n//2, random_state=42)
df_ms = ms.loc[df_ir.index]
df_ms_test = ms.loc[df_ir_test.index]

In [9]:
df_ir.reset_index(drop=True, inplace=True)
df_ir_test.reset_index(drop=True, inplace=True)
df_ms.reset_index(drop=True, inplace=True)
df_ms_test.reset_index(drop=True, inplace=True)

In [10]:
method='direct'

In [11]:
smiles_vocabs = get_or_create_smiles_vocabs(pd.concat([ms, ir]), smiles_col='SMILES', source='msd')

Loading existing character vocabulary...
SMILES vocabulary size (character): 39
Loading existing atom_wise vocabulary...
SMILES vocabulary size (atom_wise): 14
Loading existing substructure vocabulary...
SMILES vocabulary size (substructure): 3767143


In [12]:
max_mz = calculate_max_mz(pd.concat([ms]), spectrum_column='spectrum', dtype=np.array)

In [13]:
max_ir = ir['spectrum'].apply(lambda x: len(x)).max()

In [14]:
df_ms['spectrum'] = df_ms['spectrum'].apply(lambda x: bin_spectrum(x, max_mz))
df_ms_test['spectrum'] = df_ms_test['spectrum'].apply(lambda x: bin_spectrum(x, max_mz))

In [15]:
train_data = {
    'MS': df_ms,
    'IR': df_ir
}
test_data = {
    'MS': df_ms_test,
    'IR': df_ir_test
}
tokenization_methods = {
    'MS': 'direct',
    'IR': 'direct'
}
max_values = {
    'MS': max_mz,
    'IR': max_ir
}

In [16]:
train_loader, test_loader = split_and_load_tokenized_multimodal_data(
    train_data_dict=train_data,
    train_smiles=df_ms['SMILES'],
    test_data_dict=test_data,
    test_smiles=df_ms_test['SMILES'],
    tokenization_methods=tokenization_methods,
    smiles_vocab=smiles_vocabs['character'],
    max_values=max_values,
)

# sample batch used for input dimensions
sample_batch, target_batch = next(iter(train_loader))
for key in tokenization_methods.keys():
    print(key, "spectra shape:", sample_batch[key].shape)
print("SMILES shape:", target_batch.shape)
embed_depth = sample_batch['MS'].shape[3]

MS spectra shape: torch.Size([32, 1, 58, 16])
IR spectra shape: torch.Size([32, 1, 113, 16])
SMILES shape: torch.Size([32, 69])


In [17]:
smiles_vocab = smiles_vocabs['character']
smiles_vocab_size = len(smiles_vocab)

modality_configs = {
    'MS': {
        'embed_depth': 16,
        'max_length': sample_batch['MS'].shape[2],
    },
    'IR': {
        'embed_depth': 16,
        'max_length': sample_batch['IR'].shape[2],
    }
}

In [20]:
modality_configs

{'MS': {'embed_depth': 16, 'max_length': 58},
 'IR': {'embed_depth': 16, 'max_length': 113}}

In [21]:
model = MultimodalVITSeq2SeqBeam(
    smiles_vocab_size=len(smiles_vocab),
    modality_configs=modality_configs,
    d_model=64,             # 256
    nhead=4,                # 8
    num_layers=2,           # 6
    dim_feedforward=256,    # 2048
)

In [22]:
checkpoint_file = "model_checkpoints/checkpoint_8/checkpoint_epoch_50.pth"
checkpoint = torch.load(checkpoint_file)
model.load_state_dict(checkpoint['model_state_dict'])

RuntimeError: Error(s) in loading state_dict for MultimodalVITSeq2SeqBeam:
	size mismatch for length_adapters.MS.weight: copying a param with shape torch.Size([64, 60]) from checkpoint, the shape in current model is torch.Size([64, 58]).

In [20]:
ms_dict = {
    'MS': df_ms_test.sample(5),
}
ir_dict = {
    'IR': df_ir_test.sample(5),
}

In [21]:
ms_test_spectra = prepare_dataframes_for_prediction(ms_dict)
ir_test_spectra = prepare_dataframes_for_prediction(ir_dict)

In [22]:
ms_predicts = predict_smiles_from_spectra(
    model, 
    ms_test_spectra,
    tokenization_methods,
    max_values,
    smiles_vocab
)

In [23]:
ms_predicts[0]

[('1111111111111111111111111111111111111111111111111111111',
  -1.5821012258529663),
 ('SSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSS',
  -1.663003921508789),
 ('', -1.682241678237915),
 ('SSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSS',
  -2.4690637588500977),
 ('SSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSS',
  -2.5573606491088867)]

In [24]:
ir_predicts = predict_smiles_from_spectra(
    model, 
    ir_test_spectra,
    tokenization_methods,
    max_values,
    smiles_vocab
)

In [25]:
ir_predicts[0]

[('SSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSS',
  -1.8944835662841797),
 ('SSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSS',
  -2.054741621017456),
 ('', -2.251358985900879),
 ('SSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSS',
  -2.7105495929718018),
 ('///////////////////////////////////////////////////////////////////////////////////////////////////',
  -3.6777312755584717)]

In [26]:
from transformer.sequence_pred2 import is_valid_smiles, normalize_scores
from transformer.sequence_pred2 import predict_smiles_from_spectra as predict_smiles_from_spectra2
from transformer.sequence_pred2 import prepare_spectra_for_prediction as prepare_spectra_for_prediction2

In [ ]:
ms_predicts2 = {
    
}

In [ ]:
ir_predicts2 = predict_smiles_from_spectra2(
    
)